In [2]:
%pip install py3Dmol

You should consider upgrading via the '/usr/local/bin/python3.10 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
from pathlib import Path

import py3Dmol
from rdkit import Chem

def show_xyz_grid(mols, n_cols=4, width=250, height=250):
    """mols: list of RDKit Mol (3D) or .xyz path strings / Paths."""
    blocks = []
    for m in mols:
        if isinstance(m, (str, Path)):
            blocks.append(Path(m).read_text())
        else:
            blocks.append(Chem.MolToXYZBlock(m))

    n_cols = min(n_cols, len(blocks))
    n_rows = math.ceil(len(blocks) / n_cols)
    view = py3Dmol.view(
        viewergrid=(n_rows, n_cols),
        width=width * n_cols,
        height=height * n_rows,
    )
    for i, block in enumerate(blocks):
        r, c = divmod(i, n_cols)
        view.addModel(block, "xyz", viewer=(r, c))
        view.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}}, viewer=(r, c))
        view.zoomTo(viewer=(r, c))
    return view

In [4]:
import torch
from pathlib import Path
from rdkit import Chem
from tqdm import tqdm

from src.mlconfgen.egnn import EGNNDynamics
from src.mlconfgen.equivariant_flow_matching import EquivariantFlowMatching
from src.mlconfgen.utils import ATOM_DECODER, CONTEXT_NORMS
from src.mlconfgen.utils import (
    align_mol_to_principal_frame,
    prepare_edm_input,
    remove_mean_with_mask,
    samples_to_rdkit_mol,
)

# --------------------- config ---------------------
device = "mps"
HIDDEN_NF = 128
N_SAMPLES = 4
N_STEPS = 50
SEED = 42
CKPT = Path("./best_420.pt")
EDM_WEIGHTS = Path("./edm_moi_chembl_15_39.pt")  # for context_norms
MOL_PATH = "./assets/demo_files/ceyyag.mol"     # any 3D mol

torch.manual_seed(SEED)

# --------------------- load student ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}

fm_ckpt = torch.load(CKPT, map_location=device, weights_only=False)
hidden_nf = int(fm_ckpt.get("hidden_nf", HIDDEN_NF))

student = EquivariantFlowMatching(
    dynamics=EGNNDynamics(
        in_node_nf=9, context_node_nf=3, hidden_nf=hidden_nf, device=device
    ),
    in_node_nf=8,
).to(device)
student.load_state_dict(fm_ckpt["student"])
student.eval()

# --------------------- mol → context ---------------------
ref = Chem.RemoveAllHs(Chem.MolFromMolFile(MOL_PATH))
ref_context, *_ = align_mol_to_principal_frame(ref)  # (3,) MOI eigenvalues
n_atoms = ref.GetNumAtoms()

node_mask, edge_mask, context = prepare_edm_input(
    n_samples=N_SAMPLES,
    reference_context=ref_context.to(device),
    context_norms=norms,
    min_n_nodes=n_atoms,
    max_n_nodes=n_atoms,
    device=device,
)

# --------------------- FM sample (working integrator) ---------------------
@torch.inference_mode()
def sample_fm(model, node_mask, edge_mask, context, n_steps=50):
    B, N, _ = node_mask.shape
    z = model.sample_combined_position_feature_noise(B, N, node_mask)
    ts = torch.linspace(0.0, 1.0, n_steps + 1, device=z.device)

    for i in tqdm(range(n_steps)):
        t0 = ts[i].expand(B, 1)
        t1 = ts[i + 1].expand(B, 1)
        dt = (t1 - t0).view(B, 1, 1)

        # midpoint: velocity(xh, t, ...)
        v0 = model.velocity(z, t0, node_mask, edge_mask, context)
        z_mid = z + 0.5 * dt * v0
        z_mid = torch.cat(
            [remove_mean_with_mask(z_mid[..., :3], node_mask), z_mid[..., 3:]], -1
        ) * node_mask

        t_mid = 0.5 * (t0 + t1)
        v_mid = model.velocity(z_mid, t_mid, node_mask, edge_mask, context)
        z = z + dt * v_mid
        z = torch.cat(
            [remove_mean_with_mask(z[..., :3], node_mask), z[..., 3:]], -1
        ) * node_mask

    # decode: prefer all feature channels (module uses n_dims:-1; for 3+8 use n_dims:)
    x = z[:, :, : model.n_dims] * model.norm_values[0]
    h_cat = z[:, :, model.n_dims :] * model.norm_values[1] * node_mask
    h = torch.nn.functional.one_hot(
        torch.argmax(h_cat, dim=2), num_classes=model.in_node_nf
    ).float() * node_mask
    x = remove_mean_with_mask(x, node_mask)
    return x, h

x, h = sample_fm(student, node_mask, edge_mask, context, n_steps=N_STEPS)

# --------------------- tensors → RDKit (no bonds yet) ---------------------
mols = samples_to_rdkit_mol(
    positions=x.cpu(),
    one_hot=h.cpu(),
    node_mask=node_mask.cpu(),
    atom_decoder=ATOM_DECODER,
)

#Show mols
view = show_xyz_grid(mols, n_cols=4)
view.show()



100%|███████████████████████████████| 50/50 [00:08<00:00,  5.56it/s]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.